# 06 - Quantificadores e Predicados em Redes de Sensores

Este notebook formaliza a lógica de primeira ordem aplicável à rede de sensores da **Estação de Reabastecimento de Hidrogênio (SCADA-Core)**[cite: 5]. Utiliza quantificadores universais ($\forall$) e existenciais ($\exists$) para validar os estados de segurança dos Setores 100, 200 e 300 com base nos instrumentos da norma ISA-5.1[cite: 5].

---

## 1. Definição de Domínios e Predicados

### Domínios da Rede ($U$)
* **Setor 100 (Armazenamento):** $S_{100} = \{\text{PT-101}, \text{PT-102}, \text{PT-103}, \text{PT-104}, \text{PT-105}, \text{PT-106}, \text{TT-101}, \text{TT-102}, \text{TT-103}, \text{AT-101}, \text{AT-102}, \text{AT-103}\}$[cite: 5]
* **Setor 200 (Condicionamento):** $S_{200} = \{\text{TT-201}, \text{PT-201}, \text{M-201}, \text{XV-201}\}$[cite: 5]
* **Setor 300 (Dispensação):** $S_{300} = \{\text{HS-301}, \text{COM-301}, \text{BV-301}, \text{XV-301}, \text{PT-301}, \text{TT-301}, \text{AT-301}\}$[cite: 5]

### Predicados Lógicos de Processo

* **Pressão Crítica de Alarme ($P_{\text{crit}}(x)$):**
  $$P_{\text{crit}}(x) \iff \begin{cases} P(x) > 400\text{ bar}, & x = \text{PT-101} \\ P(x) > 700\text{ bar}, & x = \text{PT-102} \\ P(x) > 1000\text{ bar}, & x = \text{PT-103} \end{cases}$$[cite: 5]

* **Temperatura Crítica ($T_{\text{crit}}(x)$):**
  $$T_{\text{crit}}(x) \iff T(x) > 85^\circ\text{C}, \quad \forall x \in \{\text{TT-101}, \text{TT-102}, \text{TT-103}, \text{TT-301}\}$$[cite: 5]

* **Vazamento de Hidrogênio ($G_{\text{crit}}(x)$):**
  $$G_{\text{crit}}(x) \iff C(x) > 25\%\text{ LIE}, \quad \forall x \in \{\text{AT-101}, \text{AT-102}, \text{AT-103}, \text{AT-301}\}$$[cite: 5]

* **Pré-resfriamento Adequado ($R_{\text{ok}}(x)$):**
  $$R_{\text{ok}}(\text{TT-201}) \iff T(\text{TT-201}) \leq -40^\circ\text{C}$$[cite: 5]

## 2. Expressões Lógicas Quantificadas

### 2.1 Condição Existencial para Trip de Emergência ($\text{TRIP}_{\text{SIS}}$)
O alarme geral ($\text{ALM-101}$) e o shutdown do SIS são ativados se **existir pelo menos um** sensor no Setor 100 em estado crítico de pressão, temperatura ou gás, ou se a parada de emergência manual ($\text{ESD-100}$) for acionada[cite: 5]:

$$\text{TRIP}_{\text{SIS}} \iff \exists x \in S_{100} \big( P_{\text{crit}}(x) \lor T_{\text{crit}}(x) \lor G_{\text{crit}}(x) \big) \lor e_{1,1}$$[cite: 5]

### 2.2 Condição Universal de Integridade
O banco de cilindros do Setor 100 opera com segurança se e somente se **todos** os instrumentos do setor estiverem dentro da faixa segura[cite: 5]:

$$\text{Armazenamento}_{\text{OK}} \iff \forall x \in S_{100} \big( \neg P_{\text{crit}}(x) \land \neg T_{\text{crit}}(x) \land \neg G_{\text{crit}}(x) \big)$$[cite: 5]

### 2.3 Permissivo de Abastecimento ($\text{XV-301}$)
A abertura da válvula dispensadora $\text{XV-301}$ ($v_{3,1}$) exige integridade universal do Setor 100, pré-resfriador com $T \leq -40^\circ\text{C}$, motor $M_{201}$ operando, comunicação $COM_{301}$ ativa, trava de ruptura $BV_{301}$ conectada e comando do operador em $HS_{301}$[cite: 5]:

$$\text{Permissivo}_{\text{XV-301}} \iff \Big( \forall x \in S_{100} \big(\neg P_{\text{crit}}(x) \land \neg T_{\text{crit}}(x) \land \neg G_{\text{crit}}(x)\big) \Big) \land R_{\text{ok}}(\text{TT-201}) \land m_{2,1} \land c_{3,1} \land bv_{3,1} \land h_{3,1} \land \neg e_{1,1}$$[cite: 5]

In [1]:
# Mapeamento dos Sensores e Estados da Estacao SCADA-Core (ISA-5.1)
leituras_sensores = {
    # Setor 100: Armazenamento
    "PT-101": 380,   # bar (Limite: 400)
    "PT-102": 680,   # bar (Limite: 700)
    "PT-103": 950,   # bar (Limite: 1000)
    "PT-104": 355,   # bar (Set: >350)
    "PT-105": 655,   # bar (Set: >650)
    "PT-106": 960,   # bar (Set: >950)
    "TT-101": 42.0,  # degC (Limite: 85)
    "TT-102": 51.0,  # degC (Limite: 85)
    "TT-103": 48.0,  # degC (Limite: 85)
    "AT-101": 5.0,   # % LIE (Limite: 25)
    "AT-102": 0.0,   # % LIE (Limite: 25)
    "AT-103": 1.0,   # % LIE (Limite: 25)
    "ESD-100": False,# Botao de emergencia manual
    
    # Setor 200: Condicionamento
    "TT-201": -42.0, # degC (Limite <= -40)
    "PT-201": 700,   # bar
    "M-201": True,   # Contator Chiller Ligado
    "XV-201": True,  # Valvula de Entrada Chiller Aberta
    
    # Setor 300: Dispensacao
    "HS-301": True,  # Botao Inicio Operador
    "COM-301": True, # Comunicacao J2799 Veiculo
    "BV-301": True,  # Breakaway Integro
    "PT-301": 700,   # bar Enchimento
    "TT-301": 35.0,  # degC Veiculo (Limite: 85)
    "AT-301": 0.0    # % LIE Dispensador (Limite: 25)
}

# --- Predicados Lógicos ---

def P_crit(tag, valor):
    limites = {"PT-101": 400, "PT-102": 700, "PT-103": 1000}
    return valor > limites[tag] if tag in limites else False

def T_crit(tag, valor):
    if tag in ["TT-101", "TT-102", "TT-103", "TT-301"]:
        return valor > 85.0
    return False

def G_crit(tag, valor):
    if tag in ["AT-101", "AT-102", "AT-103", "AT-301"]:
        return valor > 25.0
    return False

def R_ok(valor_tt201):
    return valor_tt201 <= -40.0

# Quantificador Existencial: existe falha no Setor 100?
sensores_s100_criticos = [
    P_crit("PT-101", leituras_sensores["PT-101"]),
    P_crit("PT-102", leituras_sensores["PT-102"]),
    P_crit("PT-103", leituras_sensores["PT-103"]),
    T_crit("TT-101", leituras_sensores["TT-101"]),
    T_crit("TT-102", leituras_sensores["TT-102"]),
    T_crit("TT-103", leituras_sensores["TT-103"]),
    G_crit("AT-101", leituras_sensores["AT-101"]),
    G_crit("AT-102", leituras_sensores["AT-102"]),
    G_crit("AT-103", leituras_sensores["AT-103"])
]

falha_existencial_s100 = any(sensores_s100_criticos)
trip_sis = falha_existencial_s100 or leituras_sensores["ESD-100"]

# Quantificador Universal: todos os parametros seguros no Armazenamento?
armazenamento_integro = not falha_existencial_s100

# Avaliacao do Permissivo XV-301
permissivo_xv301 = all([
    armazenamento_integro,
    R_ok(leituras_sensores["TT-201"]),
    leituras_sensores["M-201"],
    leituras_sensores["COM-301"],
    leituras_sensores["BV-301"],
    leituras_sensores["HS-301"],
    not T_crit("TT-301", leituras_sensores["TT-301"]),
    not G_crit("AT-301", leituras_sensores["AT-301"]),
    not leituras_sensores["ESD-100"]
])

print("=== RELATÓRIO DE MONITORAMENTO DA REDE SCADA ===")
print(f"[SIS] Trip de Emergência Ativado: {trip_sis}")
print(f"[SETOR 100] Integridade Universal do Banco: {armazenamento_integro}")
print(f"[SETOR 200] Resfriamento (TT-201 <= -40°C): {R_ok(leituras_sensores['TT-201'])}")
print(f"[SETOR 300] Permissivo da Válvula XV-301: {permissivo_xv301}")

=== RELATÓRIO DE MONITORAMENTO DA REDE SCADA ===
[SIS] Trip de Emergência Ativado: False
[SETOR 100] Integridade Universal do Banco: True
[SETOR 200] Resfriamento (TT-201 <= -40°C): True
[SETOR 300] Permissivo da Válvula XV-301: True
